# 🔥 LangGraph + Gemini 2.5 Agent

**Goal:** Build your first AI Agent using LangGraph with Gemini!

---

## What is an AI Agent?

```
Chatbot:  User asks \u2192 AI responds
Agent:    User asks \u2192 AI thinks \u2192 uses tools \u2192 acts \u2192 observes \u2192 responds
```

**Agents can USE TOOLS - that's what makes them different!**

In [ ]:
# Install dependencies
%pip install langgraph langchain-google-genai -q

In [ ]:
# ============================================================
# 🔥 YOUR FIRST LANGGRAPH AGENT WITH GEMINI
# ============================================================

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
import os

# Load API key from .env file
load_dotenv()
API_KEY = os.environ.get("GEMINI_API_KEY")
print("\u2713 API Key loaded:", bool(API_KEY))

In [ ]:
# Create Gemini model - use gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=API_KEY)
✓✓ Gemini 2.5 Flash model created!"

## Step 1: Create Tools

Tools give your agent "hands" to interact with the world!

In [ ]:
# ============================================================
# TOOL 1: Calculator
# ============================================================
@tool
def calculator(expression: str) -> str:
    """Perform math calculations. Input: math expression like 25-8-5"""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# ============================================================
# TOOL 2: Search (using Gemini itself)
# ============================================================
@tool
def search_gemini(query: str) -> str:
    """Use this to search for information. Input: your search query."""
    response = llm.invoke(query)
    return response.content

# Create tools list
tools = [calculator, search_gemini]
✓ Tools created: calculator, search_gemini"

## Step 2: Create the Agent

Using `create_react_agent` - this is the ReAct pattern (Reason + Act)

In [ ]:
# Create LangGraph ReAct agent
agent = create_react_agent(llm, tools)
✓ LangGraph ReAct Agent created!"

## Step 3: Test the Agent!

In [ ]:
# ============================================================
# TEST 1: Math Problem (Agent uses Calculator tool)
# ============================================================
print("="*50)
print("TEST 1: Math Problem")
print("="*50)

result = agent.invoke({
    "messages": [("human", "If I have 25 apples, give 8 to John, 5 to Mary. How many apples are left?")]
})

print("\n💬 Question: If I have 25 apples, give 8 to John, 5 to Mary. How many apples are left?")
print(f"🤔 Answer: {result['messages'][-1].content}")

In [ ]:
# ============================================================
# TEST 2: Information Search (Agent uses Search tool)
# ============================================================
print("\n" + "="*50)
print("TEST 2: Information Search")
print("="*50)

result2 = agent.invoke({
    "messages": [("human", "What is RAG in machine learning?")]
})

print("\n💬 Question: What is RAG in machine learning?")
print(f"\ud83e? Answer: {result2['messages'][-1].content[:300]}...")

In [ ]:
# ============================================================
# TEST 3: Multi-step problem
# ============================================================
print("\n" + "="*50)
print("TEST 3: Multi-step Problem")
print("="*50)

result3 = agent.invoke({
    "messages": [("human", "If a train travels 120 km in 2 hours, and then 180 km in 3 hours, what is its average speed?")]
})

print("\n💬 Question: Train problem")
print(f"\ud83e? Answer: {result3['messages'][-1].content}")

---

## 🎯 ADVANCED: Custom LangGraph Workflow

In [ ]:
# ============================================================
# CUSTOM STATEGRAPH - Build your own agent workflow
# ============================================================

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# Define your custom state schema
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    question: str
    answer: str
    steps: list

# Define nodes (functions that process state)
def analyze(state: AgentState) -> AgentState:
    """Analyze the question"""
    question = state["question"]
    print(f"🔍 Analyzing: {question}")
    return {"steps": ["analyzed"]}

def generate(state: AgentState) -> AgentState:
    """Generate answer using Gemini"""
    response = llm.invoke(state["question"])
    answer = response.content
    print(f"💡 Generated answer")
    return {"answer": answer, "steps": ["generated"]}

def improve(state: AgentState) -> AgentState:
    """Improve the answer"""
    answer = state["answer"]
    improved = f"Here is a detailed answer:\n\n{answer}"
    print(f"✅ Answer improved")
    return {"answer": improved, "steps": ["improved"]}

# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("analyze", analyze)
workflow.add_node("generate", generate)
workflow.add_node("improve", improve)

# Define edges (the flow)
workflow.set_entry_point("analyze")
workflow.add_edge("analyze", "generate")
workflow.add_edge("generate", "improve")
workflow.add_edge("improve", END)

# Compile
custom_agent = workflow.compile()

# Run
result = custom_agent.invoke({
    "question": "What is a neural network in simple terms?",
    "messages": []
})

print("\n" + "="*50)
print("CUSTOM AGENT RESULT:")
print("="*50)
print(result["answer"][:300] + "...")

---

## 📊 Interview Punch

> **"I built an AI agent using LangGraph with Gemini 2.5. It's a ReAct agent that uses tools (calculator + search) to answer questions. The agent decides when to use tools by reasoning through the problem, acting on them, and observing results - this is the ReAct pattern. LangGraph lets me build complex agent workflows with custom state management - this is how production AI agents are built at companies like Databricks with their KARL agent."**

---

## 📝 Key Concepts Summary

| Concept | What It Does |
|---------|---------------|
| **LangGraph** | Build agent workflows as graphs |
| **create_react_agent** | Pre-built ReAct agent |
| **@tool decorator** | Create tools for agents |
| **StateGraph** | Custom agent with state management |
| **ReAct** | Reason + Act + Observe loop |

---

## ✅ Checkpoint

- [ ] Built LangGraph agent with Gemini
- [ ] Used @tool decorator to create tools
- [ ] Tested agent with math problem
- [ ] Tested agent with search
- [ ] Built custom StateGraph workflow
- [ ] Can explain ReAct loop to interviewer

---

*🔥 You just built a real AI Agent! This is the foundation for GCP AI Prep.*